# 02 - EDA & Statistical Analysis

## Objective
Understand what separates defaulters from non-defaulters, and quantify the
business-relevant patterns that the model will later have to reproduce.

In [ ]:
import sys
from pathlib import Path

# Make `src` importable when the notebook is run from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

from src import config, eda
from src.data import preprocessing
from src.features.engineering import add_credit_risk_features

%matplotlib inline

## Configuration and data loading

EDA runs on the cleaned frame so that impossible values do not distort the plots.

In [ ]:
raw = preprocessing.load_raw_data()
TARGET = preprocessing.detect_target(raw)
clean, cleaning_report = preprocessing.clean_data(raw, TARGET)
df = add_credit_risk_features(clean)
print(cleaning_report["duplicates_removed"], "duplicates removed")
df.shape

## Target distribution

In [ ]:
print(df[TARGET].value_counts())
print(df[TARGET].value_counts(normalize=True).round(4))
eda.plot_target_distribution(df, TARGET)

## Numeric distributions by outcome

In [ ]:
NUMERIC_FOCUS = ["person_income", "loan_amnt", "loan_int_rate",
                 "loan_percent_income", "person_age", "person_emp_length"]
eda.plot_numeric_distributions(df, NUMERIC_FOCUS, TARGET)
eda.numeric_summary_by_target(df, NUMERIC_FOCUS, TARGET)

### Significance checks

Welch t-tests on the raw values: with ~32k rows even small gaps are significant, so effect size matters more than the p-value.

In [ ]:
for column in NUMERIC_FOCUS:
    defaulted = df.loc[df[TARGET] == 1, column].dropna()
    repaid = df.loc[df[TARGET] == 0, column].dropna()
    statistic, p_value = stats.ttest_ind(defaulted, repaid, equal_var=False)
    print(f"{column:22s} t={statistic:8.2f}  p={p_value:.3g}  "
          f"mean_default={defaulted.mean():12,.1f}  mean_repaid={repaid.mean():12,.1f}")

## Default rate by categorical segment

In [ ]:
CATEGORICAL_FOCUS = ["loan_grade", "loan_intent", "person_home_ownership",
                     "age_group", "income_group", "loan_amount_group"]
eda.plot_default_rate_bars(df, CATEGORICAL_FOCUS, TARGET)

In [ ]:
for column in ["loan_grade", "loan_intent", "person_home_ownership"]:
    display(eda.default_rate_by(df, column, TARGET))

## Income, loan burden and credit history

In [ ]:
display(eda.default_rate_by(df, "income_group", TARGET))
display(eda.default_rate_by(df, "loan_amount_group", TARGET))
eda.plot_ratio_vs_default(df, "loan_percent_income", TARGET)

In [ ]:
history = df.assign(
    history_band=pd.cut(df["cb_person_cred_hist_length"], [0, 2, 5, 10, 20, 60],
                        include_lowest=True)
)
history.groupby("history_band", observed=True)[TARGET].agg(["mean", "size"]).round(4)

In [ ]:
employment = df.assign(
    employment_band=pd.cut(df["person_emp_length"], [0, 1, 3, 6, 11, 60],
                           include_lowest=True)
)
employment.groupby("employment_band", observed=True)[TARGET].agg(["mean", "size"]).round(4)

## Correlation analysis

In [ ]:
numeric_columns = [c for c in df.select_dtypes("number").columns if c != TARGET]
eda.plot_correlation_heatmap(df, numeric_columns)
df[numeric_columns + [TARGET]].corr(numeric_only=True)[TARGET].sort_values(ascending=False).round(3)

## Results and interpretation

Read the printed tables above for the exact figures. The recurring pattern across
segments is that default risk is driven by **affordability** (loan size relative to
income), by the **price of credit** (interest rate and grade, which is how the
lender already prices risk), and by **stability** (employment length, home
ownership, bureau history).

## Conclusion
The relationships are strong, monotone and economically sensible, so classical
models with good ratio features should do well -- no deep learning needed. Move on
to feature engineering (notebook 03).